# Re-evaluate trained checkpoints (no training)

Re-runs `hypdenoise.evaluate` on every `best.pt` found in the attached inputs, with the current
evaluation code (full image + field-of-view metrics), then runs `hypdenoise.compare`.

**Setup:** GPU T4 · Internet ON · Add Input → `retinal-disease-classification` **and** the Output of each
training version (Add Input → *Your Work* → notebook → output). *Save & Run All (Commit)*. ≈ 5 min per run.

In [ ]:
REPO_URL = "https://github.com/hiayoub/poincare-vs-lorentz-denoising.git"
BRANCH = "main"

In [ ]:
from pathlib import Path
hits = sorted(Path("/kaggle/input").glob("**/Training_Set"))
assert hits, "RFMiD NOT ATTACHED: Add Input -> 'retinal-disease-classification' (andrewmvd)"
ckpts = sorted(Path("/kaggle/input").glob("**/results/*_seed*/best.pt"))
print(f"RFMiD at {hits[0].parent}\n{len(ckpts)} checkpoints:")
for c in ckpts: print("  ", c.parent.name)
assert ckpts, "No checkpoints found: attach the Output of the training versions"

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
os.chdir(REPO)
print(subprocess.check_output(["git", "log", "-1", "--format=commit %h  %s"]).decode())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "third_party/hypll", "-e", "."])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips==0.1.4", "torchmetrics>=1.4"])
assert "psnr_fov" in open("src/hypdenoise/evaluate.py").read(), "evaluate.py on GitHub has no FOV metrics: push repo_patch/evaluate.py first"

In [ ]:
# Copy each run (inputs are read-only) and re-evaluate it
import shutil
OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)
seen = set()
for ck in ckpts:
    name = ck.parent.name
    if name in seen:
        print("skip duplicate", name); continue
    seen.add(name)
    dst = OUT / name
    shutil.copytree(ck.parent, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("last.pt"))
    !python -m hypdenoise.evaluate --run {dst}
    if _exit_code != 0:
        raise RuntimeError(f"Evaluation failed for {name}")

In [ ]:
!python -m hypdenoise.compare --results /kaggle/working/results --reference euclidean

Download `results/comparison.csv` and every `results/*/test_per_image.csv` + `test_summary.json` (the `.pt` files are not needed locally).